# QA — Reranking Results Inspection

Lightweight QA tool for inspecting reranking results.
Tables with product images, side-by-side baseline vs reranker, and a single NDCG comparison table.

In [22]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

from src.data.loader import load_search_data, load_product_metadata
from src.data.cleaner import clean_search_data, clean_product_metadata, normalize_text
from src.models.reranker import rerank
from src.models.strategies import RerankStrategy
from src.evaluation.metrics import evaluate_against_baseline

CDN_BASE = "https://cdn.aboutstatic.com/file"

## Load data

In [23]:
search_df = clean_search_data(load_search_data())
product_df = clean_product_metadata(load_product_metadata())
prod_lookup = product_df.set_index("product_id")
all_terms = sorted(search_df["search_term"].unique().tolist())

print(f"{len(search_df):,} rows | {len(all_terms)} terms | {len(product_df):,} products")
print(f"\nSample terms: {all_terms[:10]}")

2026-06-07 21:46:02 | src.data.loader      | INFO    | Loading search data: D:\AboutYou\data\search_term_products.parquet
2026-06-07 21:46:02 | src.data.loader      | INFO    | Loaded search_data: 19129 rows, 9 columns, 1.63MB
2026-06-07 21:46:02 | src.data.cleaner     | INFO    | Starting cleaning pipeline: 19129 rows
2026-06-07 21:46:02 | src.data.cleaner     | INFO    | Filled 102 null impression_pos values with median=26.0
2026-06-07 21:46:02 | src.data.cleaner     | INFO    | Flagged 51 rows with CTR > 100% (legitimate: bookmarks/notifications)
2026-06-07 21:46:02 | src.data.cleaner     | INFO    | Flagged 7339 rows with impressions < 3 (low confidence)
2026-06-07 21:46:02 | src.data.cleaner     | INFO    | Cleaning pipeline complete: 19129 rows
2026-06-07 21:46:02 | src.data.loader      | INFO    | Loading product metadata: D:\AboutYou\data\products.csv
2026-06-07 21:46:02 | src.data.loader      | INFO    | Loaded product_metadata: 16697 rows, 4 columns, 2.04MB
2026-06-07 21:46:0

## Baseline vs Reranker (NDCG@10)

In [24]:
evaluate_against_baseline(search_df, k=10)

2026-06-07 21:46:06 | src.evaluation.metrics | INFO    | Evaluated baseline (sort by impression_pos_avg, ascending): 305 terms, mean NDCG@10=0.4193
2026-06-07 21:46:08 | src.evaluation.metrics | INFO    | Evaluated strategy=smoothed_ctr: 305 terms, mean NDCG@10=0.8781
2026-06-07 21:46:08 | src.evaluation.metrics | INFO    | 
=== Smoothed CTR vs Baseline (NDCG@10) ===
               strategy  mean_ndcg  std_ndcg  min_ndcg  max_ndcg  n_terms
           smoothed_ctr   0.878149  0.169871  0.098039       1.0      305
baseline_impression_pos   0.419291  0.310423  0.000000       1.0      305


,strategy,mean_ndcg,std_ndcg,min_ndcg,max_ndcg,n_terms
0,smoothed_ctr,0.878149,0.169871,0.098039,1.0,305
1,baseline_impression_pos,0.419291,0.310423,0.000000,1.0,305


## Inspect reranking for a query

Shows top products as the reranker orders them, with thumbnails, scores, and the original baseline rank.

In [25]:
def image_tag(image_hash, size=64):
    if pd.isna(image_hash) or not image_hash:
        return ""
    return f"<img src='{CDN_BASE}/{image_hash}?quality=75&height={size}&width={size}' width='{size}' height='{size}'>"


def inspect(term, k=10):
    """Show reranker results with product images and baseline rank delta."""
    normalised = normalize_text(term)
    term_df = search_df[search_df["search_term"] == normalised].copy()
    if term_df.empty:
        print(f"Term '{normalised}' not in dataset.")
        print(f"\nAvailable terms ({len(all_terms)}): {all_terms[:20]} ...")
        return

    # Baseline order
    term_df = term_df.sort_values("impression_pos_avg", ascending=True).reset_index(drop=True)
    term_df["baseline_rank"] = range(1, len(term_df) + 1)

    # Reranker order
    ranked = rerank(normalised, search_df, strategy=RerankStrategy.SMOOTHED_CTR, top_k=k)

    rows = []
    for r in ranked:
        pid = r["product_id"]
        prod = prod_lookup.loc[pid] if pid in prod_lookup.index else pd.Series()
        baseline_row = term_df[term_df["product_id"] == pid]
        base_rank = int(baseline_row["baseline_rank"].iloc[0]) if not baseline_row.empty else "-"
        delta = base_rank - r["rank"] if isinstance(base_rank, int) else "-"
        delta_str = f"+{delta}" if isinstance(delta, int) and delta > 0 else str(delta)

        rows.append({
            "": image_tag(prod.get("image_hash")),
            "product": prod.get("product_name", ""),
            "rerank #": r["rank"],
            "baseline #": base_rank,
            "delta": delta_str,
            "score": r["score"],
            "clicks": r["clicks"],
            "impr.": r["impressions"],
        })

    header = f"<h3><code>{normalised}</code> — top {k} (smoothed CTR)</h3>"
    header += "<p>delta = baseline_rank - reranker_rank (positive = moved up)</p>"
    display(HTML(header + pd.DataFrame(rows).to_html(escape=False, index=False)))

### Try some queries

In [26]:
inspect("baggy jeans", k=10)

,product,rerank #,baseline #,delta,score,clicks,impr.
,jeans,1,85,+84,19.3548,89,425
,jeans,2,37,+35,12.4682,784,6256
,jeans,3,14,+11,11.9436,710,5913
,jeans,4,91,+87,11.6235,183,1543
,jeans,5,41,+36,11.1546,113,982
,jeans,6,61,+55,10.8359,139,1252
,jeans,7,90,+83,10.5623,138,1276
,jeans 'bas. baggy denim',8,15,+7,10.5234,572,5405
,jeans,9,65,+56,10.2706,185,1771
,jeans,10,8,-2,9.5624,659,6862


In [27]:
inspect("abendkleid lang", k=10)

,product,rerank #,baseline #,delta,score,clicks,impr.
,kleid,1,1,0,8.6331,47,516
,kleid,2,16,+14,8.0544,76,916
,kleid,3,46,+43,7.8406,60,738
,kleid,4,55,+51,7.3814,41,529
,abendkleid 'andrea',5,21,+16,7.3737,145,1940
,kleid,6,95,+89,7.1942,49,655
,abendkleid,7,100,+93,7.1778,43,573
,kleid,8,56,+48,7.0922,59,806
,kleid 'sanja',9,33,+24,6.8013,63,901
,kleid 'emmy',10,5,-5,6.7497,102,1486


In [28]:
inspect("adidas originals", k=10)

,product,rerank #,baseline #,delta,score,clicks,impr.
,sneaker 'samba',1,1,0,8.1673,40,462
,sneaker 'handball spezial',2,2,0,6.6792,353,5260
,sneaker 'breaknet sleek',3,25,+22,6.4083,42,631
,t-shirt '3-stripes',4,44,+40,6.2438,62,969
,hose,5,65,+60,6.0729,119,1936
,sweatjacke 'firebird',6,42,+36,5.9182,96,1599
,sneaker 'gazelle',7,78,+71,5.8198,92,1558
,sweatshirt,8,99,+91,5.7569,53,898
,hose 'essentials',9,97,+88,5.5914,51,890
,t-shirt 'britcore',10,50,+40,5.3750,42,760


### Pick your own query

Change the term below to inspect any query from the dataset.

In [36]:
inspect("adidas socken 6 paar", k=10)

,product,rerank #,baseline #,delta,score,clicks,impr.
,sportsocken 'cushioned sportswear crew 6 pairs',1,3,+2,8.8889,3,5
,übergangsjacke 'teamgeist adicolor track',2,71,+69,4.6512,1,3
,sportsocken 'cushioned crew 6 pairs',3,4,+1,4.4444,1,5
,sportsocken 'cushioned sportswear crew 6 pairs',4,2,-2,4.4444,1,5
,socken 'mid ankle 6 pairs',5,6,+1,4.4444,1,5
,unisex - strümpfe & socken 'c spw crw 3p',6,100,+94,3.1746,1,2
,sportsocken '3-stripes cushioned crew 3 pairs',7,33,+26,2.3256,0,3
,sportsocken 'performance climacool cushioned crew 3 pairs',8,32,+24,2.3256,0,3
,socken 'leopard crew 2 pairs',9,25,+16,2.3256,0,3
,sportsocken 'think linear',10,23,+13,2.3256,0,3
